# Lezione 9 — Casi d'uso reali

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 100 minuti**

## Obiettivi di apprendimento di questa lezione

- Comprendere i flussi di lavoro reali di deep learning oltre l'addestramento in notebook
- Imparare quando usare il trasferimento di apprendimento (transfer learning) invece di addestrare da zero
- Prepara modelli per la produzione: ottimizzazione, compressione, distribuzione
- Esplorare domande aperte e ricerca in corso nel deep learning

Questo notebook accompagna il capitolo 09 del manuale del corso. È l'ultimo capitolo del corso e riassume le lezioni precedenti guidandoti verso la pratica reale.

## Parte 1 — Dal notebook alla produzione

Finora hai costruito e addestrato modelli in notebook Jupyter, valutandoli su un dataset di test. In produzione, il flusso di lavoro è diverso:

1. **Raccolta e preparazione dei dati**: spesso consuma il 50-90% dello sforzo
2. **Progettazione del modello**: ora sai come farlo
3. **Addestramento**: spesso avviene offline, su un cluster GPU
4. **Ottimizzazione**: compressione, quantizzazione, pruning per velocità e memoria
5. **Distribuzione**: il modello viene caricato in un server o su dispositivo (mobile, edge)
6. **Monitoraggio**: in produzione, le prestazioni potrebbero degradarsi (data drift); il modello deve essere monitorato

Ogni fase ha sfide uniche che non hai affrontato in questo corso introattivo.

## Parte 2 — Transfer Learning

**Transfer Learning** è una tecnica che riutilizza modelli addestrati su dataset grandi per problemi nuovi con dataset piccoli.

Vuoi classificare cani di razze diverse, ma hai solo 1000 immagini di cani. Addestrare una CNN da zero su 1000 immagini è rischioso: alta probabilità di sovradattamento.

Invece:

1. Scarica una CNN già addestrata su ImageNet (1.2 milioni di immagini di migliaia di categorie), come ResNet o EfficientNet
2. Sostituisci l'ultimo strato (che predice 1000 categorie ImageNet) con uno nuovo che predice il tuo problema (es. 10 razze di cani)
3. Congela i pesi dei primi strati (il modello già sa riconoscere forme e texture)
4. Addestra solo l'ultimo strato (o "fine-tuna" gli ultimi 1-2 strati) sui tuoi 1000 dati di cani

Risultato: veloce, accurato e con meno dati. La rete ha già imparato feature generali da ImageNet e le applica al tuo problema.

## Parte 3 — Modelli preaddestrati in PyTorch

`torchvision.models` contiene modelli CNN preaddestrati su ImageNet, pronti all'uso:

```python
from torchvision.models import resnet50
modello = resnet50(pretrained=True)  # scarica pesi preaddestrati
```

Questi modelli sono il punto di partenza per la maggior parte dei problemi di visione artificiale in produzione.

## Parte 4 — Ottimizzazione per la produzione

Un modello "raw" addestrato è raramente pronto per la produzione:

- **Quantizzazione**: converti i pesi da float32 (4 byte) a int8 (1 byte), riducendo le dimensioni del modello del 4× con perdita di accuratezza minima
- **Pruning**: rimuovi i pesi piccoli (prossimi a zero), il modello usa meno memoria e computazione
- **Distillazione della conoscenza**: addestra un modello piccolo per replicare il comportamento di uno grande
- **ONNX export**: converti PyTorch in un formato standard per distribuire su varie piattaforme (TensorFlow, MobileNet, browser, ecc.)

PyTorch ha strumenti integrati per quantizzazione e ONNX; il pruning è un po' più manuale.

Perché? In un'app mobile o un serverless (dove il tempo di risposta è critico), un modello più piccolo e veloce è essenziale.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. TENSORES: transformamos cada imagen a un tensor antes de usarla
transformaciones = transforms.Compose([transforms.ToTensor()])

# 2. DATASET: reutilizamos MNIST, igual que en las clases 2, 3, 4 y 7
dataset_entrenamiento = datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
dataset_test = datasets.MNIST(
    root="./datos", train=False, download=True, transform=transformaciones
)

# Nos quedamos con un subconjunto pequeño para que el repaso sea rápido:
# el objetivo de esta celda es repasar el patrón, no maximizar la precisión
subconjunto_entrenamiento = torch.utils.data.Subset(dataset_entrenamiento, range(2000))
cargador_entrenamiento = DataLoader(subconjunto_entrenamiento, batch_size=128, shuffle=True)

print("Ejemplos de entrenamiento usados en este repaso:", len(subconjunto_entrenamiento))
print("Ejemplos de test disponibles:", len(dataset_test))


# 3. MODELO: la misma red densa de tres capas de las clases 3, 4 y 7
class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # aplana cada imagen 28x28 en un vector de 784 valores
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        return self.capa3(x)           # logits en bruto: CrossEntropyLoss se encarga del resto


modelo = RedMNIST()
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.001)

# 4. ENTRENAMIENTO: el mismo bucle de la clase 4, aquí solo dos épocas
modelo.train()
for epoca in range(2):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        optimizador.zero_grad()
        salida = modelo(imagenes)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_epoca += perdida.item()
    print(f"Época {epoca + 1}: pérdida media = {perdida_epoca / len(cargador_entrenamiento):.3f}")

# 5. EVALUACIÓN: predecimos sobre dos ejemplos de test que el modelo no ha visto
modelo.eval()
with torch.no_grad():
    for indice in [0, 1]:
        imagen, etiqueta_real = dataset_test[indice]
        salida = modelo(imagen.unsqueeze(0))   # unsqueeze añade la dimensión de lote (batch)
        prediccion = torch.argmax(salida, dim=1).item()
        print(f"Ejemplo {indice}: etiqueta real = {etiqueta_real}, predicción del modelo = {prediccion}")

## Parte 5 — Quando addestrare da zero vs. transfer learning

**Addestra da zero quando:**

- Hai un dataset grande (> 100k immagini) e specifico al dominio
- Il tuo problema è molto diverso da ImageNet (es. immagini mediche a raggi X, immagini satellitari)
- Vuoi un'architettura personalizzata

**Usa transfer learning quando:**

- Hai meno di 100k immagini
- Il tuo dominio è simile a ImageNet (fotografie naturali, oggetti, volti)
- Hai risorse limitate (tempo, GPU, denaro)

La maggior parte dei progetti reali usa transfer learning.

## Parte 6 — Problemi aperti e ricerca

Il deep learning risolve molti problemi, ma molti rimangono aperti:

- **Robustezza**: un modello che funziona perfettamente su MNIST fallisce con immagini leggermente rumorose (attacchi avversari)
- **Interpretabilità**: perché il modello ha fatto quella predizione? ("black box problem")
- **Efficienza**: come addestrare modelli sempre più grandi con meno energia?
- **Few-shot learning**: come imparare da pochissimi esempi, come una persona?
- **Reinforcement learning**: come insegnare a un agente a giocare o controllare un robot?

Questi sono i confini della ricerca contemporanea e le opportunità per innovazione.